# Knowledge Distillation Demo — ORDA vs torch.compile Comparison

This notebook allows you to run a **side-by-side comparison** of ORDA vs. standard `torch.compile` baseline under identical conditions (same seed, same model, same data, same hyperparameters).

Run each cell **in order** in Google Colab or Kaggle (GPU runtime).

* **Cell 3** runs the distillation using **ORDA** (`orda_ce_kernel` Triton kernel).
* **Cell 4** runs the distillation using standard **`torch.compile` Baseline**.

After both runs, compare the **DEMO SUMMARY** at the bottom of each cell for VRAM and throughput (`tok/s`).

## Cell 0 — Install

In [ ]:
!pip install -q "transformers>=4.45.0" "datasets>=2.18" "accelerate>=0.27"
!pip install -q git+https://github.com/hiwuhgds-pixel/ORDA-Knowledge-Distillation-Kernel.git

import importlib
for pkg in ("transformers", "datasets", "accelerate", "orda_ce_kernel"):
    try:
        importlib.import_module(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — install failed, check network/logs")


## Cell 1 — Setup Model & Tokenizer

In [ ]:
# Colab/Kaggle workaround: mock torchvision before other imports to avoid
# "operator torchvision::nms does not exist" errors from unsloth/transformers.
import sys, importlib.machinery
from types import ModuleType

class MockModule(ModuleType):
    def __init__(self, name):
        super().__init__(name)
        self.__path__ = []
    def __getattr__(self, name):
        if name.startswith("__") and name.endswith("__"):
            raise AttributeError(name)
        full_name = f"{self.__name__}.{name}"
        if full_name not in sys.modules:
            sub = MockModule(full_name)
            sub.__spec__ = importlib.machinery.ModuleSpec(full_name, None, origin="mock")
            sys.modules[full_name] = sub
        return sys.modules[full_name]

for _n in ["torchvision", "torchvision.io", "torchvision.ops",
           "torchvision.transforms", "torchvision.transforms.functional",
           "torchvision.transforms.v2"]:
    _m = MockModule(_n)
    _m.__spec__ = importlib.machinery.ModuleSpec(_n, None, origin="mock")
    sys.modules[_n] = _m

import gc, math, os, time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

from orda_ce_kernel import (
    distillation_loss,
    SeparateTeacher,
    KernelConfig,
    is_available as orda_available,
)

# ── Device ──────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    gpu_vram = props.total_memory / 1024**3
else:
    gpu_name = "CPU"
    gpu_vram = 0.0

print(f"ORDA Triton available: {orda_available()}")

# ── Load Teacher ────────────────────────────────────────────
TEACHER_ID = "unsloth/Llama-3.2-1B"
print(f"Loading tokenizer: {TEACHER_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

VOCAB_SIZE   = len(tokenizer)
PAD_TOKEN_ID = tokenizer.pad_token_id

print(f"\nLoading teacher: {TEACHER_ID} ...")
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

TEACHER_HIDDEN = teacher.config.hidden_size   # 2048
teacher_head_w = teacher.lm_head.weight       # [vocab, 2048]

# ── Student Model Definition ────────────────────────────────
@dataclass
class StudentConfig:
    hidden    : int = 512
    n_layers  : int = 12
    q_heads   : int = 8
    kv_heads  : int = 2
    head_dim  : int = 64
    ffn_dim   : int = 1536
    max_seq   : int = 2048
    rope_base : int = 500_000

STUDENT_CFG = StudentConfig()

def _precompute_rope(head_dim, max_seq, base=500_000):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)
    rope  = torch.cat([freqs, freqs], dim=-1)
    return rope.cos(), rope.sin()

def _rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def _apply_rope(q, k, cos, sin):
    T = q.shape[-2]
    c, s = cos[:T].unsqueeze(0).unsqueeze(0), sin[:T].unsqueeze(0).unsqueeze(0)
    return q * c + _rotate_half(q) * s, k * c + _rotate_half(k) * s

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.w   = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, Hq, Hk, Dh = cfg.hidden, cfg.q_heads, cfg.kv_heads, cfg.head_dim
        self.Hq, self.Hk, self.Dh, self.kv_group = Hq, Hk, Dh, Hq // Hk
        self.q_proj = nn.Linear(d, Hq * Dh, bias=False)
        self.k_proj = nn.Linear(d, Hk * Dh, bias=False)
        self.v_proj = nn.Linear(d, Hk * Dh, bias=False)
        self.o_proj = nn.Linear(Hq * Dh, d, bias=False)
    def _repeat_kv(self, x):
        if self.kv_group == 1: return x
        B, Hk, T, Dh = x.shape
        return x.unsqueeze(2).expand(B, Hk, self.kv_group, T, Dh).reshape(B, Hk * self.kv_group, T, Dh)
    def forward(self, x, cos, sin):
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.Hq, self.Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.Hk, self.Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.Hk, self.Dh).transpose(1, 2)
        q, k = _apply_rope(q, k, cos, sin)
        k, v = self._repeat_kv(k), self._repeat_kv(v)
        return self.o_proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D))

class SwiGLU(nn.Module):
    def __init__(self, d, ff):
        super().__init__()
        self.gate = nn.Linear(d, ff, bias=False)
        self.up   = nn.Linear(d, ff, bias=False)
        self.down = nn.Linear(ff, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.hidden)
        self.ffn_norm  = RMSNorm(cfg.hidden)
        self.attn      = GQAttention(cfg)
        self.ffn       = SwiGLU(cfg.hidden, cfg.ffn_dim)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x

class StudentLM(nn.Module):
    def __init__(self, vocab, cfg):
        super().__init__()
        self.cfg     = cfg
        self.embed   = nn.Embedding(vocab, cfg.hidden)
        self.blocks  = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.hidden)
        self.lm_head = nn.Linear(cfg.hidden, vocab, bias=False)
        self.lm_head.weight = self.embed.weight   # tied weights
        cos, sin = _precompute_rope(cfg.head_dim, cfg.max_seq, cfg.rope_base)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        self._init_weights()
    def _init_weights(self):
        std = 0.02
        nn.init.normal_(self.embed.weight, std=std)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.weight is not self.embed.weight:
                nn.init.normal_(m.weight, std=std / math.sqrt(2 * self.cfg.n_layers))
    def forward(self, input_ids, return_hidden=False):
        h = self.embed(input_ids)
        for block in self.blocks:
            h = block(h, self.cos, self.sin)
        h = self.norm(h)
        return h if return_hidden else self.lm_head(h)
    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())

student = StudentLM(VOCAB_SIZE, STUDENT_CFG).to(device=device, dtype=torch.float32)

# ── AdamW Weight Decay Helper ──────────────────────────────
def _param_groups(model, wd):
    no_decay, decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim == 1 or "norm" in name or "bias" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay,    "weight_decay": wd},
            {"params": no_decay, "weight_decay": 0.0}]

# ── Print KD Setup Summary ───────────────────────────────────
setup_lines = [
    f"Teacher Model : {TEACHER_ID} ({sum(p.numel() for p in teacher.parameters())/1e6:.1f}M params)",
    f"Student Model : StudentLM ({student.n_params/1e6:.1f}M params)",
    f"Vocab Size    : {VOCAB_SIZE:,} tokens (Pad: {PAD_TOKEN_ID})",
    f"Hardware      : {gpu_name} (VRAM: {gpu_vram:.1f} GB | AMP: float16)"
]
max_len = max(len(line) for line in setup_lines)
box_width = max_len + 6

print("┌" + "─" * (box_width - 2) + "┐")
print(f"│ {'KNOWLEDGE DISTILLATION SETUP':^{box_width - 4}} │")
print("├" + "─" * (box_width - 2) + "┤")
for line in setup_lines:
    print(f"│  {line:<{box_width - 6}}  │")
print("└" + "─" * (box_width - 2) + "┘")


## Cell 2 — Dataset & DataLoader Prep

In [ ]:
# ── Hyperparams ──────────────────────────────────────────────
SEED           = 42
SEQ_LEN        = 512
BATCH          = 8
GRAD_ACCUM     = 8
LR             = 3e-4
LR_MIN         = 3e-5
WARMUP_STEPS   = 30
MAX_STEPS      = 200
WEIGHT_DECAY   = 0.1
GRAD_CLIP      = 1.0
LOG_EVERY      = 20
SAVE_EVERY     = 300
CKPT_DIR       = "./checkpoints_comparison"
os.makedirs(CKPT_DIR, exist_ok=True)

KD_TEMPERATURE = 2.0
KD_WEIGHT      = 0.5
CE_WEIGHT      = 1.0

# Cosine decay schedule helper
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * step / max(WARMUP_STEPS, 1)
    progress = (step - WARMUP_STEPS) / max(MAX_STEPS - WARMUP_STEPS, 1)
    return LR_MIN + (LR - LR_MIN) * 0.5 * (1 + math.cos(math.pi * progress))

# ── Load Dataset ────────────────────────────────────────────
from datasets import load_dataset
DATASET_NAME   = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"

print(f"Loading dataset: {DATASET_NAME}/{DATASET_CONFIG} ...")
raw = load_dataset(DATASET_NAME, DATASET_CONFIG)

class ChunkDataset(Dataset):
    def __init__(self, chunks):
        self.data = [torch.tensor(c, dtype=torch.long) for c in chunks]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        x = self.data[idx]
        return x[:-1], x[1:]

def tokenize_and_chunk(split_name, seq_len):
    full_text = "\n\n".join(t for t in raw[split_name]["text"] if t.strip())
    token_ids = tokenizer.encode(full_text, add_special_tokens=False)
    bos       = tokenizer.bos_token_id or 0
    chunks    = [[bos] + token_ids[i : i + seq_len - 1]
                 for i in range(0, len(token_ids) - seq_len, seq_len)]
    print(f"  {split_name}: {len(token_ids):,} tokens → {len(chunks):,} chunks")
    return chunks

print("Preparing tokenized chunks...")
train_chunks = tokenize_and_chunk("train",      SEQ_LEN)
val_chunks   = tokenize_and_chunk("validation", SEQ_LEN)


## Cell 3 — Train with ORDA

In [ ]:
print("  " + "─" * 60)
print("  RUN 1: Distillation using ORDA Kernel")
print("  " + "─" * 60)

# Reseed and recreate dataloader to ensure identical batch ordering
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

_g = torch.Generator()
_g.manual_seed(SEED)
train_loader = DataLoader(ChunkDataset(train_chunks), batch_size=BATCH,
                          shuffle=True, pin_memory=True, drop_last=True, generator=_g)
val_loader   = DataLoader(ChunkDataset(val_chunks),   batch_size=BATCH,
                          shuffle=False, pin_memory=True, drop_last=False)

# Recreate the full student for a clean run, including RMSNorm weights
student = StudentLM(VOCAB_SIZE, STUDENT_CFG).to(device=device, dtype=torch.float32)

# Reset CUDA peak memory stats
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

# ORDA configuration
kernel_cfg = KernelConfig(
    online_softmax=True,
    fast_math=False,
    quantize_grad_weight=False,
    stochastic_rounding=False,
    fp32_grad_weight_accumulation=False,
    max_chunks=None,
)

optimizer = torch.optim.AdamW(
    _param_groups(student, WEIGHT_DECAY),
    lr=get_lr(1),
    betas=(0.9, 0.95),
    eps=1e-8,
    fused=(device.type == "cuda"),
)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def get_teacher_outputs(input_ids):
    with torch.no_grad():
        out = teacher(input_ids.to(teacher.device),
                      output_hidden_states=True, use_cache=False)
    h_t = out.hidden_states[-1].to(device=device, dtype=DTYPE)
    h_t = h_t.reshape(-1, TEACHER_HIDDEN).contiguous()
    w_t = teacher_head_w.to(device=device, dtype=DTYPE)
    return h_t, w_t

@torch.compile(mode="default", disable=(device.type != "cuda"))
def student_forward_orda(model, input_ids):
    return model(input_ids, return_hidden=True)

def train_step_orda(input_ids, labels):
    h_teacher, w_teacher = get_teacher_outputs(input_ids)
    with torch.autocast(device_type=device.type, dtype=DTYPE):
        h_student   = student_forward_orda(student, input_ids)
        h_student   = h_student.reshape(-1, STUDENT_CFG.hidden).contiguous()
        w_student   = student.lm_head.weight
        labels_flat = labels.reshape(-1)

        out = distillation_loss(
            h_student,
            w_student,
            labels_flat,
            SeparateTeacher(h_teacher, w_teacher),
            student_ce_weight=CE_WEIGHT,
            teacher_ce_weight=0.0,
            kd_weight=KD_WEIGHT,
            temperature=KD_TEMPERATURE,
            ignore_index=PAD_TOKEN_ID,
            backend="auto",
            config=kernel_cfg,
        )
    return {"loss": out.loss, "student_ce": out.student_ce.item(), "kl": out.kl.item()}

def validate(loader, max_batches=50):
    student.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches: break
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=DTYPE):
                loss = F.cross_entropy(student(x).reshape(-1, VOCAB_SIZE),
                                       y.reshape(-1), ignore_index=PAD_TOKEN_ID)
            total += loss.item(); n += 1
    student.train()
    return total / max(n, 1)

student.train()
global_step    = 0
accum_loss     = accum_ce = accum_kl = 0.0
peak_vram_mb   = 0.0
step_times     = []
last_avg_loss  = float("nan")
last_tok_per_s = 0.0

optimizer.zero_grad(set_to_none=True)
train_iter = iter(train_loader)

while global_step < MAX_STEPS:
    global_step += 1
    lr_now = get_lr(global_step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr_now

    t_step = time.time()

    for _ in range(GRAD_ACCUM):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)
        x, y = x.to(device), y.to(device)
        result = train_step_orda(x, y)
        scaler.scale(result['loss'] / GRAD_ACCUM).backward()
        accum_loss += result['loss'].item()
        accum_ce   += result["student_ce"]
        accum_kl   += result["kl"]

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    step_times.append(time.time() - t_step)

    if global_step % LOG_EVERY == 0:
        avg_loss       = accum_loss / (LOG_EVERY * GRAD_ACCUM)
        avg_ce         = accum_ce   / (LOG_EVERY * GRAD_ACCUM)
        avg_kl         = accum_kl   / (LOG_EVERY * GRAD_ACCUM)
        avg_step_s     = sum(step_times) / len(step_times)
        tok_per_s      = GRAD_ACCUM * BATCH * (SEQ_LEN - 1) / avg_step_s
        last_avg_loss  = avg_loss
        last_tok_per_s = tok_per_s

        if device.type == "cuda":
            cur_vram     = torch.cuda.max_memory_allocated() / 1024**2
            peak_vram_mb = max(peak_vram_mb, cur_vram)
            vram_mb      = f"{cur_vram:>7.0f}"
        else:
            vram_mb      = "    N/A"

        print(
            f"update={global_step:>5d}/{MAX_STEPS:<5d} | "
            f"loss={avg_loss:>8.4f} | "
            f"ce={avg_ce:>8.4f} | "
            f"kl={avg_kl:>8.4f} | "
            f"lr={lr_now:>9.2e} | "
            f"speed={tok_per_s:>7.0f} tok/s | "
            f"alloc_peak={vram_mb} MB"
        )
        accum_loss = accum_ce = accum_kl = 0.0
        step_times = []

# Final validation & Save
val_loss = validate(val_loader)
ckpt_path = os.path.join(CKPT_DIR, "student_orda_final.pt")
torch.save({"model_state": student.state_dict()}, ckpt_path)
print(f"\n  ↳ Saved checkpoint: {ckpt_path}")

# ── Print Demo Summary Box ───────────────────────────────────
gpu_model_str = f"{torch.cuda.get_device_name(0)} (Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB)" if device.type == "cuda" else "CPU"
summary_lines = [
    f"GPU Model   : {gpu_model_str}",
    f"Peak Tensor Alloc : {peak_vram_mb:.0f} MB",
    f"Throughput  : {last_tok_per_s:.0f} tok/s",
    f"Final Loss  : {last_avg_loss:.4f}"
]
max_len = max(len(line) for line in summary_lines)
box_width = max_len + 6

print("\n┌" + "─" * (box_width - 2) + "┐")
print(f"│ {'Demo Summary (ORDA Run)':^{box_width - 4}} │")
print("├" + "─" * (box_width - 2) + "┤")
for line in summary_lines:
    print(f"│  {line:<{box_width - 6}}  │")
print("└" + "─" * (box_width - 2) + "┘")

# ── Clean up variables to free VRAM for Baseline Run ───────
del optimizer, scaler
gc.collect()
torch.cuda.empty_cache()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()


## Cell 4 — Train with torch.compile Baseline

In [ ]:
print("  " + "─" * 60)
print("  RUN 2: Distillation using torch.compile Baseline")
print("  " + "─" * 60)

# Reseed and recreate dataloader to ensure identical batch ordering
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

_g = torch.Generator()
_g.manual_seed(SEED)
train_loader = DataLoader(ChunkDataset(train_chunks), batch_size=BATCH,
                          shuffle=True, pin_memory=True, drop_last=True, generator=_g)
val_loader   = DataLoader(ChunkDataset(val_chunks),   batch_size=BATCH,
                          shuffle=False, pin_memory=True, drop_last=False)

# Recreate the full student for a clean run, including RMSNorm weights
student = StudentLM(VOCAB_SIZE, STUDENT_CFG).to(device=device, dtype=torch.float32)

# Reset CUDA peak memory stats
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

optimizer = torch.optim.AdamW(
    _param_groups(student, WEIGHT_DECAY),
    lr=get_lr(1),
    betas=(0.9, 0.95),
    eps=1e-8,
    fused=(device.type == "cuda"),
)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def get_teacher_outputs(input_ids):
    with torch.no_grad():
        out = teacher(input_ids.to(teacher.device),
                      output_hidden_states=True, use_cache=False)
    h_t = out.hidden_states[-1].to(device=device, dtype=DTYPE)
    h_t = h_t.reshape(-1, TEACHER_HIDDEN).contiguous()
    w_t = teacher_head_w.to(device=device, dtype=DTYPE)
    return h_t, w_t

# Compile baseline forward and loss region
@torch.compile(mode="default", disable=(device.type != "cuda"))
def forward_and_loss_baseline(student_model, input_ids, labels_flat, h_teacher, w_teacher):
    h_student   = student_model(input_ids, return_hidden=True)
    h_student   = h_student.reshape(-1, STUDENT_CFG.hidden).contiguous()
    w_student   = student_model.lm_head.weight.to(dtype=h_student.dtype)

    logits_s = F.linear(h_student, w_student)
    logits_t = F.linear(h_teacher, w_teacher)

    loss_ce  = F.cross_entropy(logits_s, labels_flat, ignore_index=PAD_TOKEN_ID)

    log_p_s  = F.log_softmax(logits_s / KD_TEMPERATURE, dim=-1)
    p_t      = F.softmax((logits_t / KD_TEMPERATURE).detach(), dim=-1)
    mask     = (labels_flat != PAD_TOKEN_ID).unsqueeze(-1)
    loss_kl  = (F.kl_div(log_p_s, p_t, reduction="none") * mask).sum() \
               / torch.clamp(mask.sum(), min=1) * (KD_TEMPERATURE ** 2)

    return CE_WEIGHT * loss_ce + KD_WEIGHT * loss_kl, loss_ce, loss_kl

def train_step_baseline(input_ids, labels):
    h_teacher, w_teacher = get_teacher_outputs(input_ids)
    labels_flat = labels.reshape(-1)
    with torch.autocast(device_type=device.type, dtype=DTYPE):
        loss, loss_ce, loss_kl = forward_and_loss_baseline(
            student, input_ids, labels_flat, h_teacher, w_teacher)
    return {"loss": loss, "student_ce": loss_ce.item(), "kl": loss_kl.item()}

def validate(loader, max_batches=50):
    student.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches: break
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=DTYPE):
                loss = F.cross_entropy(student(x).reshape(-1, VOCAB_SIZE),
                                       y.reshape(-1), ignore_index=PAD_TOKEN_ID)
            total += loss.item(); n += 1
    student.train()
    return total / max(n, 1)

student.train()
global_step    = 0
accum_loss     = accum_ce = accum_kl = 0.0
peak_vram_mb   = 0.0
step_times     = []
last_avg_loss  = float("nan")
last_tok_per_s = 0.0

optimizer.zero_grad(set_to_none=True)
train_iter = iter(train_loader)

try:
    while global_step < MAX_STEPS:
        global_step += 1
        lr_now = get_lr(global_step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr_now

        t_step = time.time()

        for _ in range(GRAD_ACCUM):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                x, y = next(train_iter)
            x, y = x.to(device), y.to(device)
            result = train_step_baseline(x, y)
            scaler.scale(result['loss'] / GRAD_ACCUM).backward()
            accum_loss += result['loss'].item()
            accum_ce   += result["student_ce"]
            accum_kl   += result["kl"]

        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step_times.append(time.time() - t_step)

        if global_step % LOG_EVERY == 0:
            avg_loss       = accum_loss / (LOG_EVERY * GRAD_ACCUM)
            avg_ce         = accum_ce   / (LOG_EVERY * GRAD_ACCUM)
            avg_kl         = accum_kl   / (LOG_EVERY * GRAD_ACCUM)
            avg_step_s     = sum(step_times) / len(step_times)
            tok_per_s      = GRAD_ACCUM * BATCH * (SEQ_LEN - 1) / avg_step_s
            last_avg_loss  = avg_loss
            last_tok_per_s = tok_per_s

            if device.type == "cuda":
                cur_vram     = torch.cuda.max_memory_allocated() / 1024**2
                peak_vram_mb = max(peak_vram_mb, cur_vram)
                vram_mb      = f"{cur_vram:>7.0f}"
            else:
                vram_mb      = "    N/A"

            print(
                f"update={global_step:>5d}/{MAX_STEPS:<5d} | "
                f"loss={avg_loss:>8.4f} | "
                f"ce={avg_ce:>8.4f} | "
                f"kl={avg_kl:>8.4f} | "
                f"lr={lr_now:>9.2e} | "
                f"speed={tok_per_s:>7.0f} tok/s | "
                f"alloc_peak={vram_mb} MB"
            )
            accum_loss = accum_ce = accum_kl = 0.0
            step_times = []

    # Final validation & Save
    val_loss = validate(val_loader)
    ckpt_path = os.path.join(CKPT_DIR, "student_baseline_final.pt")
    torch.save({"model_state": student.state_dict()}, ckpt_path)

except torch.cuda.OutOfMemoryError:
    oom_lines = [
        f"Status      : Stopped at step {global_step}/{MAX_STEPS}",
        f"Peak Tensor Alloc : {torch.cuda.max_memory_allocated()/1024**2:.0f} MB",
        "",
        "Technical Cause:",
        "The standard PyTorch KL path materializes full logits tensors",
        "of shape [Batch * Seq, Vocab] in FP16, leading to high memory pressure."
    ]
    max_len = max(len(line) for line in oom_lines)
    box_width = max_len + 6
    print("\n┌" + "─" * (box_width - 2) + "┐")
    print(f"│ {'Baseline Run OOM Error':^{box_width - 4}} │")
    print("├" + "─" * (box_width - 2) + "┤")
    for line in oom_lines:
        print(f"│  {line:<{box_width - 6}}  │")
    print("└" + "─" * (box_width - 2) + "┘")
    peak_vram_mb = torch.cuda.max_memory_allocated() / 1024**2

# ── Print Demo Summary Box ───────────────────────────────────
gpu_model_str = f"{torch.cuda.get_device_name(0)} (Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB)" if device.type == "cuda" else "CPU"
summary_lines = [
    f"GPU Model   : {gpu_model_str}",
    f"Peak Tensor Alloc : {peak_vram_mb:.0f} MB",
    f"Throughput  : {last_tok_per_s:.0f} tok/s" if last_tok_per_s > 0 else "Throughput  : 0 tok/s (OOM)",
    f"Final Loss  : {last_avg_loss:.4f}" if last_tok_per_s > 0 else "Final Loss  : None"
]
max_len = max(len(line) for line in summary_lines)
box_width = max_len + 6

print("\n┌" + "─" * (box_width - 2) + "┐")
print(f"│ {'Demo Summary (Baseline Run)':^{box_width - 4}} │")
print("├" + "─" * (box_width - 2) + "┤")
for line in summary_lines:
    print(f"│  {line:<{box_width - 6}}  │")
print("└" + "─" * (box_width - 2) + "┘")
